# How to use
- เปิด Playground mode ด้วยการกด `File` -> `Open in playground mode`
- พิมพ์เลขปีงบประมาณของงบที่ต้องการสร้าง csv ในช่อง `เลขปีงบประมาณ`
- copy url ของ Google Sheet ข้อมูลงบที่ validate แล้วมาใส่ในช่อง `url ของ Google Sheet Tree งบประมาณที่ validate แล้ว`
- copy url ของ Google Sheet ที่ต้องการสร้างตารางผลลัพธ์มาใส่ในช่อง `url ของ Google Sheet ที่ต้องการบันทึกผลลัพธ์`
- กด Run all
---
Note
- ในการรันครั้งแรกระบบจะขอให้ login ด้วย Google Account
- Login ด้วย account ที่มี permission แก้ไข [sheet สำหรับ validate](https://docs.google.com/spreadsheets/d/1zArSJZE1l3nuOMbPvnxNGmATLy1g1e0IjBcJkxJLFxU/edit?gid=51205943#gid=51205943?target=_blank)

In [1]:
#@title เลขปีงบประมาณ
BUDGET_YEAR = 2570 #@param {type:"integer"}

In [2]:
#@title url ของ Google Sheet Tree งบประมาณที่ validate แล้ว
TREE_SHEET_URL = 'https://docs.google.com/spreadsheets/d/1zArSJZE1l3nuOMbPvnxNGmATLy1g1e0IjBcJkxJLFxU/edit?gid=331654191#gid=331654191' # @param {"type": "string"}

In [3]:
#@title url ของ Google Sheet ที่ต้องการบันทึกผลลัพธ์
OUTPUT_SHEET_URL = 'https://docs.google.com/spreadsheets/d/1tr4SZDs0Si4OFePFF5ZiaEwbDhaV4XgG5m6lE8FBjig/edit?gid=1843464751#gid=1843464751' # @param {"type": "string"}

In [4]:
#@title อัปโหลดไฟล์
#@markdown Run cell นี้และกด `Choose Files` เพื่ออัปโหลดไฟล์ doc_url_index.json

from google.colab import files
import os, json

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "Uploading_Data_Colab_1.xlsx" with length 9000 bytes'.format(
      name=fn, length=len(uploaded[fn])))

assert os.path.exists('doc_url_index.json'), 'Please upload doc_url_index.json'

with open("doc_url_index.json", "r") as f:
  PDF_NAME_INDEX = json.load(f)

Saving doc_url_index.json to doc_url_index.json
User uploaded file "Uploading_Data_Colab_1.xlsx" with length 9000 bytes


# Code

In [5]:
import os
import re
import requests
import json
import pandas as pd
import datetime

## Authenticate to Google

In [6]:
# Connect to Google Account & Google Sheet
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

## Connect to Google Sheet

In [7]:
gc = gspread.authorize(creds)

In [8]:
# Get validation tree sheet
validate_sheet = gc.open_by_url(TREE_SHEET_URL)
validate_sheet

<Spreadsheet '[70/3] Budget Tree For Validation' id:1zArSJZE1l3nuOMbPvnxNGmATLy1g1e0IjBcJkxJLFxU>

In [9]:
# Get output sheet
output_sheet = gc.open_by_url(OUTPUT_SHEET_URL)
output_sheet

<Spreadsheet '[70/3] Thailand_s Budget' id:1tr4SZDs0Si4OFePFF5ZiaEwbDhaV4XgG5m6lE8FBjig>

## Clone & Install Validation Tool

In [10]:
!rm -rf wevis-openbudget-validation-tools && git clone https://github.com/wevisdemo/wevis-openbudget-validation-tools.git

Cloning into 'wevis-openbudget-validation-tools'...
remote: Enumerating objects: 487, done.
remote: Counting objects: 100% (487/487), done.
remote: Compressing objects: 100% (292/292), done.
remote: Total 487 (delta 242), reused 427 (delta 182), pack-reused 0 (from 0)
Receiving objects: 100% (487/487), 96.54 KiB | 2.30 MiB/s, done.
Resolving deltas: 100% (242/242), done.


In [11]:
!pip install -e "wevis-openbudget-validation-tools/open-budget-tree-validator" -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for open_budget_tree_validator (pyproject.toml) ... done


In [12]:
import site

for sitepackages in site.getsitepackages():
    site.addsitedir(sitepackages)

# Build Final CSV

## Load ministry names

In [13]:
ministry_name_df = pd.read_csv("https://raw.githubusercontent.com/wevisdemo/wevis-openbudget-ocr/refs/heads/master/data/min_agc_names.csv")
ministry_name_df.head(5)

,min_name,agc_name
0,สำนักนายกรัฐมนตรี,สำนักงานปลัดสำนักนายกรัฐมนตรี
1,สำนักนายกรัฐมนตรี,กรมประชาสัมพันธ์
2,สำนักนายกรัฐมนตรี,สำนักงานคณะกรรมการคุ้มครองผู้บริโภค
3,สำนักนายกรัฐมนตรี,สำนักเลขาธิการนายกรัฐมนตรี
4,สำนักนายกรัฐมนตรี,สำนักเลขาธิการคณะรัฐมนตรี


In [14]:
ministries_name = ministry_name_df['min_name'].unique()
ministries_name

array(['สำนักนายกรัฐมนตรี', 'กระทรวงกลาโหม', 'กระทรวงการคลัง',
       'กระทรวงการต่างประเทศ', 'กระทรวงการท่องเที่ยวและกีฬา',
       'กระทรวงการพัฒนาสังคมและความมั่นคงของมนุษย์',
       'กระทรวงเกษตรและสหกรณ์', 'กระทรวงคมนาคม',
       'กระทรวงทรัพยากรธรรมชาติและสิ่งแวดล้อม',
       'กระทรวงดิจิทัลเพื่อเศรษฐกิจและสังคม', 'กระทรวงพลังงาน',
       'กระทรวงพาณิชย์', 'กระทรวงมหาดไทย', 'กระทรวงยุติธรรม',
       'กระทรวงแรงงาน', 'กระทรวงวัฒนธรรม', 'กระทรวงศึกษาธิการ',
       'กระทรวงสาธารณสุข', 'กระทรวงอุตสาหกรรม',
       'กระทรวงการอุดมศึกษา วิทยาศาสตร์ วิจัยและนวัตกรรม',
       'ส่วนราชการไม่สังกัดสำนักนายกรัฐมนตรี กระทรวง หรือทบวง และหน่วยงานภายใต้การควบคุมดูแลของนายกรัฐมนตรี',
       'หน่วยงานของรัฐสภา', 'หน่วยงานของศาล',
       'หน่วยงานขององค์กรอิสระและองค์กรอัยการ', 'รัฐวิสาหกิจ',
       'ส่วนราชการในพระองค์', 'สภากาชาดไทย', 'หน่วยงานอื่นของรัฐ',
       'จังหวัดและกลุ่มจังหวัด', 'องค์กรปกครองส่วนท้องถิ่น',
       'ทุนหมุนเวียน', 'กองทุนและเงินทุนหมุนเวียน', 'งบกลาง',
       'รายจ่ายเพื่

## Construct PDF name Index

In [15]:
doc_name_index = {
    _doc['pdf_filename']:_doc['doc_title'] for _doc in PDF_NAME_INDEX
}
doc_name_index

{'1790738384_5610.pdf': 'เอกสารงบประมาณ ฉบับที่ ๓ งบประมาณรายจ่าย ฉบับปรับปรุง ตามพระราชบัญญัติงบประมาณรายจ่าย ประจำปีงบประมาณ พ.ศ. ๒๕๗๐ เล่มที่ ๑',
 '1790738860_7240.pdf': 'เอกสารงบประมาณ ฉบับที่ ๓ งบประมาณรายจ่าย ฉบับปรับปรุง ตามพระราชบัญญัติงบประมาณรายจ่าย ประจำปีงบประมาณ พ.ศ. ๒๕๗๐ เล่มที่ ๒',
 '1790738926_6118.pdf': 'เอกสารงบประมาณ ฉบับที่ ๓ งบประมาณรายจ่าย ฉบับปรับปรุง ตามพระราชบัญญัติงบประมาณรายจ่าย ประจำปีงบประมาณ พ.ศ. ๒๕๗๐ เล่มที่ ๓ (๑)',
 '1790739048_2775.pdf': 'เอกสารงบประมาณ ฉบับที่ ๓ งบประมาณรายจ่าย ฉบับปรับปรุง ตามพระราชบัญญัติงบประมาณรายจ่าย ประจำปีงบประมาณ พ.ศ. ๒๕๗๐ เล่มที่ ๓ (๒)',
 '1790739126_9372.pdf': 'เอกสารงบประมาณ ฉบับที่ ๓ งบประมาณรายจ่าย ฉบับปรับปรุง ตามพระราชบัญญัติงบประมาณรายจ่าย ประจำปีงบประมาณ พ.ศ. ๒๕๗๐ เล่มที่ ๓ (๓)',
 '1790739534_4834.pdf': 'เอกสารงบประมาณ ฉบับที่ ๓ งบประมาณรายจ่าย ฉบับปรับปรุง ตามพระราชบัญญัติงบประมาณรายจ่าย ประจำปีงบประมาณ พ.ศ. ๒๕๗๐ เล่มที่ ๓ (๔)',
 '1790739606_3550.pdf': 'เอกสารงบประมาณ ฉบับที่ ๓ งบประมาณรายจ่าย ฉบับปรับปรุง ตามพระราชบั

## Build DF for each Ministry

In [16]:
from open_budget_tree_validator import BudgetTree

In [17]:
import re

def assign_doc_name(doc: str) -> str:
  doc_name = doc_name_index.get(doc, None)
  if doc_name is None:
    return ''

  doc_issue_number = re.search(r"ฉบับที่ \d", doc_name).group(0)
  doc_number = re.search(r"(เล่มที่ \d+(\s?\(\d+\))?)", doc_name).group(1)

  formatted_doc_name = f"เอกสารงบประมาณ {doc_issue_number} {doc_number}"

  # Convert Thai number to Arabic
  thai_to_arabic_map = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')

  return formatted_doc_name.translate(thai_to_arabic_map)

In [18]:
def assign_url(doc: str, page: int) -> str:

  BASE_URL = "https://bbstore.bb.go.th/cms/"

  page_url = f"{BASE_URL}{doc}?#page={page}"

  return page_url

In [19]:
def get_budget_plan_info(budget_plan: str) -> tuple[str, str]:
  # BUDGET_PLAN_PREFIX & BUDGET_PLAN_SUFFIX
  PREFIXES_LIST = [
      'แผนงานบริหารจัดการหนี้ภาครัฐ', 'แผนงานบริหารเพื่อรองรับกรณีฉุกเฉินหรือจำเป็น',
      'แผนงานบุคลากรภาครัฐ', 'แผนงานบูรณาการ', 'แผนงานพื้นฐาน',
      'แผนงานยุทธศาสตร์', 'แผนงานรายจ่ายเพื่อชดใช้เงินคงคลัง'
  ]
  match_pattern = r"^(" + '|'.join(PREFIXES_LIST) + r")((.*)?$)"
  budget_plan_match = re.search(match_pattern, budget_plan)
  if not budget_plan_match:
    return '', ''

  return budget_plan_match.group(1), budget_plan_match.group(2)

In [20]:
import numpy as np

added_ministries = set()

final_df = None
for sheet in validate_sheet.worksheets():
  sheet_title = sheet.title

  matched_ministry_name = re.search(
      r"(" + "|".join(ministries_name) + ")",
      sheet_title
  )
  if matched_ministry_name and not re.search(r"\[", sheet_title):
    if matched_ministry_name.group(1) not in added_ministries:

      print(f"process {sheet_title} ...")

      # Load data from Sheet
      sheet_values = sheet.get_all_values()
      df = pd.DataFrame(sheet_values[1:], columns=sheet_values[0])

      # Normalize: for any row without document; copy from previous row that have
      df["document"] = df["document"].replace(
          {
              "": np.nan,
              " ": np.nan
          }
      ).ffill()

      budget_tree = BudgetTree(budget_tree_df=df, budget_year=BUDGET_YEAR)
      added_ministries.add(matched_ministry_name.group(1))

      if final_df is None:
        final_df = budget_tree.build_final_df(BUDGET_YEAR)
        continue
      final_df = pd.concat([
          final_df,
          budget_tree.build_final_df(BUDGET_YEAR)
      ])


    final_df[['BUDGET_PLAN_PREFIX', 'BUDGET_PLAN_SUFFIX']] = final_df.apply(
        lambda row: get_budget_plan_info(row['BUDGET_PLAN']),
        axis=1,
        result_type='expand'
    )

    # Add pdf doc name & url
    final_df['PDF_NAME'] = final_df['_document'].apply(assign_doc_name)
    final_df['PAGE_URL'] = final_df.apply(
        lambda row: assign_url(row['_document'], row['_page']),
        axis=1
    )


process กระทรวงการอุดมศึกษา วิทยาศาสตร์ วิจัยและนวัตกรรม ...
process กระทรวงการท่องเที่ยวและกีฬา ...
process กระทรวงมหาดไทย ...
process กระทรวงดิจิทัลเพื่อเศรษฐกิจและสังคม ...
process กระทรวงพลังงาน ...
process กระทรวงศึกษาธิการ ...
process กระทรวงวัฒนธรรม ...
process งบกลาง ...
process กระทรวงการคลัง ...
process กระทรวงการพัฒนาสังคมและความมั่นคงของมนุษย์ ...
process กระทรวงพาณิชย์ ...
process กระทรวงแรงงาน ...
process กระทรวงอุตสาหกรรม ...
process กระทรวงการต่างประเทศ ...
process กระทรวงทรัพยากรธรรมชาติและสิ่งแวดล้อม ...
process กระทรวงยุติธรรม ...
process จังหวัดและกลุ่มจังหวัด ...
process กระทรวงเกษตรและสหกรณ์ ...
process กระทรวงคมนาคม ...
process กระทรวงสาธารณสุข ...
process หน่วยงานอื่นของรัฐ ...
process ส่วนราชการในพระองค์ ...
process ส่วนราชการไม่สังกัดสำนักนายกรัฐมนตรี กระทรวง หรือทบวง และหน่วยงานภายใต้การควบคุมดูแลของนายกรัฐมนตรี ...
process รัฐวิสาหกิจ ...
process สำนักนายกรัฐมนตรี ...
process สภากาชาดไทย ...
process รายจ่ายเพื่อชดใช้เงินคงคลัง ...
process ทุนหมุนเวียน ...
pr

In [21]:
final_df.head(5)

,REF_DOC,REF_PAGE_NO,MINISTRY,BUDGETARY_UNIT,BUDGET_PLAN,CROSS_FUNC?,OUTPUT,PROJECT,CATEGORY_LV1,CATEGORY_LV2,...,ITEM_DESCRIPTION,AMOUNT,FISCAL_YEAR,OBLIGED?,_document,_page,BUDGET_PLAN_PREFIX,BUDGET_PLAN_SUFFIX,PDF_NAME,PAGE_URL
0,1790738926_6118.pdf,63,กระทรวงการอุดมศึกษา วิทยาศาสตร์ วิจัยและนวัตกรรม,สำนักงานปลัดกระทรวงการอุดมศึกษา วิทยาศาสตร์ วิ...,แผนงานบุคลากรภาครัฐ,False,,,งบบุคลากร,เงินเดือนและค่าจ้างประจำ,...,เงินเดือน,273522700.0,2027,False,1790738926_6118.pdf,63,แผนงานบุคลากรภาครัฐ,,เอกสารงบประมาณ ฉบับที่ 3 เล่มที่ 3 (1),https://bbstore.bb.go.th/cms/1790738926_6118.p...
1,1790738926_6118.pdf,63,กระทรวงการอุดมศึกษา วิทยาศาสตร์ วิจัยและนวัตกรรม,สำนักงานปลัดกระทรวงการอุดมศึกษา วิทยาศาสตร์ วิ...,แผนงานบุคลากรภาครัฐ,False,,,งบบุคลากร,เงินเดือนและค่าจ้างประจำ,...,ค่าจ้างประจํา,11881800.0,2027,False,1790738926_6118.pdf,63,แผนงานบุคลากรภาครัฐ,,เอกสารงบประมาณ ฉบับที่ 3 เล่มที่ 3 (1),https://bbstore.bb.go.th/cms/1790738926_6118.p...
2,1790738926_6118.pdf,63,กระทรวงการอุดมศึกษา วิทยาศาสตร์ วิจัยและนวัตกรรม,สำนักงานปลัดกระทรวงการอุดมศึกษา วิทยาศาสตร์ วิ...,แผนงานบุคลากรภาครัฐ,False,,,งบบุคลากร,,...,ค่าจ้างชั่วคราว,3744100.0,2027,False,1790738926_6118.pdf,63,แผนงานบุคลากรภาครัฐ,,เอกสารงบประมาณ ฉบับที่ 3 เล่มที่ 3 (1),https://bbstore.bb.go.th/cms/1790738926_6118.p...
3,1790738926_6118.pdf,63,กระทรวงการอุดมศึกษา วิทยาศาสตร์ วิจัยและนวัตกรรม,สำนักงานปลัดกระทรวงการอุดมศึกษา วิทยาศาสตร์ วิ...,แผนงานบุคลากรภาครัฐ,False,,,งบบุคลากร,,...,ค่าตอบแทนพนักงานราชการ,15502700.0,2027,False,1790738926_6118.pdf,63,แผนงานบุคลากรภาครัฐ,,เอกสารงบประมาณ ฉบับที่ 3 เล่มที่ 3 (1),https://bbstore.bb.go.th/cms/1790738926_6118.p...
4,1790738926_6118.pdf,63,กระทรวงการอุดมศึกษา วิทยาศาสตร์ วิจัยและนวัตกรรม,สำนักงานปลัดกระทรวงการอุดมศึกษา วิทยาศาสตร์ วิ...,แผนงานบุคลากรภาครัฐ,False,,,งบดำเนินงาน,ค่าตอบแทน ใช้สอยและวัสดุ,...,ค่าเช่าบ้าน,5776900.0,2027,False,1790738926_6118.pdf,63,แผนงานบุคลากรภาครัฐ,,เอกสารงบประมาณ ฉบับที่ 3 เล่มที่ 3 (1),https://bbstore.bb.go.th/cms/1790738926_6118.p...


In [22]:
# Drop any columns that start with `_`
final_df = final_df.loc[:, ~final_df.columns.str.startswith('_')]

# Add new Sheet

In [23]:
import zoneinfo

bangkok_tz = zoneinfo.ZoneInfo("Asia/Bangkok")
now_bangkok = datetime.datetime.now(bangkok_tz)

sheet_title = now_bangkok.date().isoformat() + '#'
sheet_title += now_bangkok.strftime("%H:%M:%S")

new_sheet = output_sheet.add_worksheet(
    title=sheet_title,
    rows=len(final_df.index),
    cols=len(final_df.columns)
)
new_sheet.update(
    [final_df.columns.values.tolist()] + final_df.values.tolist(),
    value_input_option='USER_ENTERED'
)

{'spreadsheetId': '1tr4SZDs0Si4OFePFF5ZiaEwbDhaV4XgG5m6lE8FBjig',
 'updatedRange': "'2026-10-29#20:13:32'!A1:W66871",
 'updatedRows': 66871,
 'updatedColumns': 23,
 'updatedCells': 1538033}